# 🧩 Mini-Lab: Explaining Your Trade-offs

**Module 11: Ship It: Capstone, Docs, Responsible AI** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Answer** four interview questions using numbers and files from your own repo
2. **Reject** answers that say "looks good" instead of naming a metric or a failure mode
3. **Outline** an optional 90-second demo: what it does, one command, one number, one limitation

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Evidence-Based Decision | "I would ship V2 because ___, accepting ___" is an evidence-based launch call that ties measured quality, latency, and cost to explicit risk tradeoffs. |
| Interface Seam | A small typed contract between your app and an AI vendor. Your adapter returns your dataclass, never their SDK response object, so providers can be swapped without rewrites. |
| Percentile Latency | The mean hides the slow calls and timeouts, so use percentiles like p50 and p95 to describe typical latency and tail slowness users actually feel. |
| AI Failure Modes | Common ways AI apps fail in the real world: out-of-schema labels, data and user drift, rate limits, silent fallbacks, and input drift that degrade quality, latency, or cost without obvious errors. |


## Four questions, four rejected answers

Fill the blanks from your repo if you have different numbers. The structure stays.


In [1]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))


In [2]:
METRICS = {
    "gold_n": 50,
    "baseline_acc": 0.62,
    "billing_precision": 0.84,
    "billing_recall": 0.71,
    "p50_ms": 420,
    "p95_ms": 2100,
    "cost_per_1000": 0.42,
}

QUESTIONS = [
    {
        "id": "Q1",
        "ask": "Why is the model behind an interface instead of a vendor SDK call in the pipeline?",
        "rejected": "Interfaces are best practice and more professional.",
        "hired": (
            "Pipeline code types a Protocol and receives our dataclass. Tests inject FakeClient "
            "so timeout and bad JSON do not need a network or a key. Swapping hosted vs local "
            "is a constructor argument, not a rewrite."
        ),
        "concept": "Interface seam",
    },
    {
        "id": "Q2",
        "ask": "Would you ship this over the baseline?",
        "rejected": "Yes, the model looks more accurate.",
        "hired": (
            f"I would ship this as a demonstration Billing router because Billing precision is "
            f"{METRICS['billing_precision']:.0%} on a {METRICS['gold_n']}-row gold set "
            f"(baseline accuracy {METRICS['baseline_acc']:.0%}), accepting that Account is often "
            f"labelled General and that this is not a production system."
        ),
        "concept": "Evidence-based decision",
    },
    {
        "id": "Q3",
        "ask": "What does p95 hide that the mean does not, and what do 1,000 records cost?",
        "rejected": "It is pretty fast and cheap enough.",
        "hired": (
            f"p50 is {METRICS['p50_ms']} ms; p95 is {METRICS['p95_ms']} ms, so a tail of slow calls "
            f"(and timeouts) is hiding behind the mean. Cost per 1,000 records is "
            f"${METRICS['cost_per_1000']:.2f} from logged call counts, with price treated as a variable."
        ),
        "concept": "Percentile latency",
    },
    {
        "id": "Q4",
        "ask": "Name two production failure modes this system hits first, and what you would do.",
        "rejected": "LLMs sometimes hallucinate, so we would monitor it.",
        "hired": (
            "First: out-of-schema labels ('Refunds') — validate at the boundary and fail the row. "
            "Second: silent fallback to General on parse errors — structured log with parse_error, "
            "never coerce. Rate-limit 429s are third: bounded retry, then a non-zero CLI exit."
        ),
        "concept": "AI failure modes",
    },
]

blocks = []
for q in QUESTIONS:
    blocks.append(
        f"### {q['id']} — {q['concept']}\n\n"
        f"**Question.** {q['ask']}\n\n"
        f"**Rejected.** {q['rejected']}\n\n"
        f"**Hired.** {q['hired']}"
    )
md("\n\n".join(blocks))


### Q1 — Interface seam

**Question.** Why is the model behind an interface instead of a vendor SDK call in the pipeline?

**Rejected.** Interfaces are best practice and more professional.

**Hired.** Pipeline code types a Protocol and receives our dataclass. Tests inject FakeClient so timeout and bad JSON do not need a network or a key. Swapping hosted vs local is a constructor argument, not a rewrite.

### Q2 — Evidence-based decision

**Question.** Would you ship this over the baseline?

**Rejected.** Yes, the model looks more accurate.

**Hired.** I would ship this as a demonstration Billing router because Billing precision is 84% on a 50-row gold set (baseline accuracy 62%), accepting that Account is often labelled General and that this is not a production system.

### Q3 — Percentile latency

**Question.** What does p95 hide that the mean does not, and what do 1,000 records cost?

**Rejected.** It is pretty fast and cheap enough.

**Hired.** p50 is 420 ms; p95 is 2100 ms, so a tail of slow calls (and timeouts) is hiding behind the mean. Cost per 1,000 records is $0.42 from logged call counts, with price treated as a variable.

### Q4 — AI failure modes

**Question.** Name two production failure modes this system hits first, and what you would do.

**Rejected.** LLMs sometimes hallucinate, so we would monitor it.

**Hired.** First: out-of-schema labels ('Refunds') — validate at the boundary and fail the row. Second: silent fallback to General on parse errors — structured log with parse_error, never coerce. Rate-limit 429s are third: bounded retry, then a non-zero CLI exit.

## Optional 90-second demo outline

Not required to pass. If you record it, keep this order so you do not spend the minute on architecture theatre.


In [3]:
DEMO = [
    ("0:00", "What it does", "This CLI labels support tickets Billing, Technical, Account, or General."),
    ("0:20", "One command", "python -m triage.cli data/support_inbox.csv"),
    ("0:45", "One number", "Billing precision 84% on 50 gold rows; baseline 62%; p95 2100 ms."),
    ("1:10", "One limitation", "Demonstration project, not production. Account vs General is the weak cut."),
]

lines = ["| Time | Beat | Line |", "|---|---|---|"]
for t, beat, line in DEMO:
    lines.append(f"| {t} | {beat} | {line} |")
md("### Demo beats\n\n" + "\n".join(lines))


### Demo beats

| Time | Beat | Line |
|---|---|---|
| 0:00 | What it does | This CLI labels support tickets Billing, Technical, Account, or General. |
| 0:20 | One command | python -m triage.cli data/support_inbox.csv |
| 0:45 | One number | Billing precision 84% on 50 gold rows; baseline 62%; p95 2100 ms. |
| 1:10 | One limitation | Demonstration project, not production. Account vs General is the weak cut. |

## 🎯 Summary

### Key Takeaways

1. **Evidence-based decision** — ship or do not ship names a gold-set number and an accepted miss
2. **Interface seam** — the interview answer is "tests inject a fake," not "we used a Protocol because SOLID"
3. **Percentile latency** — p95 plus cost per 1,000 beats "fast and cheap"
4. **AI failure modes** — pick two that this pipeline actually hits (schema, silent fallback, 429)
